In [1]:
import pandas as pd

Data loading

In [2]:
TRAIN_PATH = "/Users/david/Documents/data/jusbrasil/manual/with_body/splits/train.parquet"
VAL_PATH = "/Users/david/Documents/data/jusbrasil/manual/with_body/splits/vali.parquet"
#TEST_PATH = "/Users/david/Documents/data/jusbrasil/manual/with_body/splits/small_test.parquet"
TEST_PATH = "/Users/david/Documents/data/jusbrasil/manual/with_body/splits/vali.parquet"

In [3]:
train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VAL_PATH)
test_df = pd.read_parquet(TEST_PATH)

Prepocessing

In [4]:
def preprocess_llms(df):
   return df[["query","title","body","court","relevance"]]

In [5]:
train_df = preprocess_llms(train_df)
test_df = preprocess_llms(test_df)
val_df = preprocess_llms(val_df)

In [6]:
train_df['body'].str.len().describe()

count      9188.000000
mean      20108.203200
std       24784.232236
min           0.000000
25%        8397.500000
50%       14141.500000
75%       23782.750000
max      449686.000000
Name: body, dtype: float64

Evaluating by LLMs

In [7]:
from pathlib import Path
import os

from dotenv import load_dotenv

load_dotenv(Path("/Users/david/Documents/dev/llm-as-judge-for-ltr/.env"))

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "").strip()
if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY is missing. Put it in the project .env file.")

In [8]:
import os
import pandas as pd
from openai import OpenAI
from functools import cmp_to_key

# client = OpenAI(
#     base_url="https://openrouter.ai/api/v1",    
#     api_key=os.getenv("OPENROUTER_API_KEY"),
# )


#Connect to Ollama
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

In [9]:
# MODEL = "qwen/qwen-2.5-72b-instruct"
# MODEL = "meta-llama/llama-3.3-70b-instruct"


#OLLAMA MODELS

#MODEL = "gemma2:2b"
MODEL = "llama3.1:8b"
#MODEL = "meta-llama/llama-3.3-70b-instruct"
#MODEL = "qwen/qwen-2.5-72b-instruct"

In [10]:
import time
import string


def format_doc(row, max_chars=1500):
    title = str(row.get("title", "Sem título"))
    court = str(row.get("court", "Tribunal Nao Informado"))
    body = str(row.get("body", ""))[:max_chars]
    return f"Title: {title}\nCourt: {court}\nText: {body}..."

def compare_documents_setwise(docs, query_text):
    """
    Compara um conjunto de k documentos simultaneamente.
    Retorna o índice (0, 1, 2...) do documento considerado mais relevante.
    """
    system_prompt = "You are an expert Brazilian legal information retrieval judge. Your task is to compare multiple documents and determine which one is the most relevant to a user's search query."
    
    # Gera rótulos A, B, C, D... de acordo com o tamanho de docs
    labels = list(string.ascii_uppercase)[:len(docs)]
    
    # Formata os documentos no prompt
    docs_formatted = []
    for label, doc in zip(labels, docs):
        docs_formatted.append(f"Document {label}:\n{format_doc(doc)}")
    
    docs_string = "\n\n".join(docs_formatted)
    
    # Cria a string de opções válidas (ex: 'A', 'B', or 'C')
    if len(labels) > 1:
        valid_labels = ", ".join([f"'{l}'" for l in labels[:-1]]) + f", or '{labels[-1]}'"
    else:
        valid_labels = f"'{labels[0]}'"

    user_prompt = f"""Query: {query_text}

{docs_string}

Which document is the most relevant to the query? Output exactly the letter of the most relevant document ({valid_labels}). Do not provide any explanation or extra text."""

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.0,
            max_tokens=2,
        )
        
        winner_token = response.choices[0].message.content.strip().upper()
        
        # Encontra qual rótulo foi retornado e devolve o índice original
        for i, label in enumerate(labels):
            if label in winner_token:
                return i 
                
        # Retorna -1 se houver alucinação ou resposta fora do padrão
        return -1 
        
    except Exception as e:
        print(f"Error comparing setwise documents: {e}")
        return -1

In [11]:
# Original Setwise (Zhuang et al., SIGIR 2024): 1 LLM call per set.
# Circular permutation / majority vote is not part of the main method.

In [12]:
import pandas as pd
from tqdm.auto import tqdm


def pick_best_index(docs, query_text):
    """One LLM call. Returns the index of the most relevant document."""
    if len(docs) == 1:
        return 0
    winner_idx = compare_documents_setwise(docs, query_text)
    if winner_idx < 0 or winner_idx >= len(docs):
        return 0
    return winner_idx


def heapify(arr, n, i, query_text, num_child, call_counter):
    """Compare parent + children in one setwise call (Zhuang et al. / ielab llm-rankers)."""
    child_start = num_child * i + 1
    if child_start >= n:
        return
    child_end = min(num_child * (i + 1) + 1, n)
    inds = [i] + list(range(child_start, child_end))
    subset = [arr[j] for j in inds]
    call_counter[0] += 1
    best_local = pick_best_index(subset, query_text)
    largest = inds[best_local]
    if largest != i:
        arr[i], arr[largest] = arr[largest], arr[i]
        heapify(arr, n, largest, query_text, num_child, call_counter)


def setwise_heapsort(docs, query_text, k=4, top_n=None):
    """
    Max-heap setwise sort. k is docs per LLM call (c in Zhuang et al.).
    Each heapify compares the parent plus (k-1) children, so one call covers a k-set.
    Default k=4 matches pairwise body budget: 1500*4 == 3000*2.
    top_n=None ranks the full list; top_n=10 matches the paper's early stop.
    Returns (best-first docs, n_llm_calls).
    """
    arr = list(docs)
    n = len(arr)
    if n <= 1:
        return arr, 0

    num_child = max(k - 1, 1)
    call_counter = [0]
    extract_n = n if top_n is None else min(top_n, n)

    for i in range(n // num_child, -1, -1):
        heapify(arr, n, i, query_text, num_child, call_counter)

    ranked = 0
    for i in range(n - 1, 0, -1):
        arr[i], arr[0] = arr[0], arr[i]
        ranked += 1
        if ranked == extract_n:
            break
        heapify(arr, i, 0, query_text, num_child, call_counter)

    if extract_n == n:
        ordered = list(reversed(arr))
    else:
        ordered = list(reversed(arr[n - extract_n:])) + arr[: n - extract_n]
    return ordered, call_counter[0]


def run_setwise_pilot(test_df, k=4, top_n=None):
    """
    Setwise heapsort reranker (Zhuang et al.).

    Args:
        test_df: DataFrame with query/title/court/body/relevance
        k: documents compared in each LLM call (default 4 = same ~6k body chars as pairwise)
        top_n: extract only the first N (paper uses 10). None = full ranking.
    """
    results = []
    grouped = list(test_df.groupby("query"))
    total_calls = 0

    for query_text, group in tqdm(grouped, total=len(grouped), desc="Ranking queries (Setwise heapsort)"):
        docs = group.to_dict(orient="records")
        ordered, n_calls = setwise_heapsort(docs, query_text, k=k, top_n=top_n)
        total_calls += n_calls
        for rank, doc in enumerate(ordered, start=1):
            row = doc.copy()
            row["llm_rank"] = rank
            results.append(row)

    n_queries = max(len(grouped), 1)
    print(f"LLM calls: {total_calls} total, {total_calls / n_queries:.1f} per query")
    return pd.DataFrame(results)

/Users/david/Library/Caches/pypoetry/virtualenvs/llm-as-judge-for-ltr-UctXQ_0n-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
df_llm_ranked = run_setwise_pilot(test_df)
print("Ranking complete.")

Ranking queries (Setwise heapsort): 100%|██████████| 109/109 [5:04:34<00:00, 167.66s/it] 


LLM calls: 3528 total, 32.4 per query
Ranking complete.


In [14]:
import numpy as np

def ndcg_at_k(labels_sorted_by_score, labels_ideal, k):
    k = min(k, len(labels_sorted_by_score))
    predicted = labels_sorted_by_score[:k]
    ideal = np.sort(labels_ideal)[::-1][:k]
    discounts = np.log2(np.arange(1, k + 1) + 1)
    dcg = np.sum((np.power(2.0, predicted) - 1) / discounts)
    idcg = np.sum((np.power(2.0, ideal) - 1) / discounts)
    return 0.0 if idcg == 0.0 else dcg / idcg

ks = [1, 3, 5, 10]
rows = []

for query, g in df_llm_ranked.groupby("query"):
    y_q = g["relevance"].to_numpy(dtype=float)
    # rank 1 first (ascending), unlike LambdaMART which sorts score descending
    y_ranked = y_q[np.argsort(g["llm_rank"].to_numpy())]

    row = {"query": query, "num_docs": len(g)}
    for k in ks:
        row[f"ndcg@{k}"] = ndcg_at_k(y_ranked, y_q, k)
    rows.append(row)

metrics_llm = pd.DataFrame(rows)
for k in ks:
    print(f"LLM  NDCG@{k:<3} = {metrics_llm[f'ndcg@{k}'].mean():.4f}")

LLM  NDCG@1   = 0.8008
LLM  NDCG@3   = 0.7979
LLM  NDCG@5   = 0.7915
LLM  NDCG@10  = 0.8019


In [15]:
from pathlib import Path

RESULTS_PATH = Path("/Users/david/Documents/dev/llm-as-judge-for-ltr/notebooks/data/PoC_results_setwise.csv")


def append_poc_results(model, metrics_df, notes=""):
    row = {
        "model": model,
        "ndcg@1": round(metrics_df["ndcg@1"].mean(), 4),
        "ndcg@3": round(metrics_df["ndcg@3"].mean(), 4),
        "ndcg@5": round(metrics_df["ndcg@5"].mean(), 4),
        "ndcg@10": round(metrics_df["ndcg@10"].mean(), 4),
        "n_queries": len(metrics_df),
        "notes": notes,
    }
    new = pd.DataFrame([row])
    if RESULTS_PATH.exists():
        out = pd.concat([pd.read_csv(RESULTS_PATH), new], ignore_index=True)
    else:
        out = new
    out.to_csv(RESULTS_PATH, index=False)
    return out


In [16]:
append_poc_results(MODEL, metrics_llm, notes="setwise LLM judge on small_test")

,model,ndcg@1,ndcg@3,ndcg@5,ndcg@10,n_queries,notes
0,google/gemma-4-26b-a4b-it,0.9032,0.9002,0.8878,0.8850,30,pairwise LLM judge on small_test k=3
1,google/gemma-4-26b-a4b-it,0.9032,0.8846,0.8791,0.8773,30,pairwise LLM judge on small_test k=4
2,gemma2:9b,0.8476,0.8438,0.8529,0.8569,30,pairwise LLM judge on small_test
3,gemma2:2b,0.8143,0.7570,0.7837,0.8183,30,setwise LLM judge on small_test
4,llama3.1:8b,0.8365,0.8342,0.8093,0.8356,30,setwise LLM judge on small_test
5,meta-llama/llama-3.3-70b-instruct,0.8698,0.8584,0.8406,0.8542,30,setwise LLM judge on small_test
6,qwen/qwen-2.5-72b-instruct,0.8587,0.8599,0.8470,0.8585,30,setwise LLM judge on small_test
7,llama3.1:8b,0.8008,0.7979,0.7915,0.8019,109,setwise LLM judge on small_test
